# Pricer Project: runnable module tour

Run top to bottom with the project kernel after `uv sync --locked`. Core examples use local synthetic data and need no credentials, network, or model service. Optional live work is shown separately.

## Setup and package map

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').exists(): raise RuntimeError('Open this notebook from the project checkout')
SRC = ROOT / 'src'
if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))
print('Project:', ROOT, '| Python:', sys.executable)

Project: C:\Users\aksha\PycharmProjects\PricerProject | Python: C:\Users\aksha\PycharmProjects\PricerProje\venv\Scripts\python.exe


In [2]:
# Inventory of the package areas and modules covered by this notebook.
MODULES = {
 'data': 'items parser loaders batch',
 'agents': 'agent deals guardrails retrieval planning_agent specialist_agent scanner_agent ensemble_agent state_graph llm_provider messaging_agent frontier_agent autonomous_planning_agent rag_judge deal_seen_store neural_network_agent deep_neural_network preprocessor',
 'application': 'framework', 'inference': 'qwen finetuned',
 'persistence': 'intelligence_store', 'telemetry': 'monitoring observability',
 'evaluation': 'evaluator', 'experiments': 'config data baselines plots human neural local_llm',
 'gateway_evals': 'runner', 'interfaces': 'api dashboard dependencies log_utils routes schemas mcp_server mcp_client',
 'jobs': 'build_bm25 train_neural train_dpo evaluate_dpo gateway_eval',
 'integrations.modal': 'hello service service_cached'
}
for area, names in MODULES.items(): print(f'{area}: {names}')
print('Total modules:', sum(len(names.split()) for names in MODULES.values()))

data: items parser loaders batch
agents: agent deals guardrails retrieval planning_agent specialist_agent scanner_agent ensemble_agent state_graph llm_provider messaging_agent frontier_agent autonomous_planning_agent rag_judge deal_seen_store neural_network_agent deep_neural_network preprocessor
application: framework
inference: qwen finetuned
persistence: intelligence_store
telemetry: monitoring observability
evaluation: evaluator
experiments: config data baselines plots human neural local_llm
gateway_evals: runner
interfaces: api dashboard dependencies log_utils routes schemas mcp_server mcp_client
jobs: build_bm25 train_neural train_dpo evaluate_dpo gateway_eval
integrations.modal: hello service service_cached
Total modules: 53


## Product data, parsing, and configuration

In [3]:
from pricer.config import Settings, runtime_home
from pricer.data.items import Item
from pricer.data.parser import get_weight, scrub, parse
settings = Settings.from_env()
description = scrub('Example insulated bottle', ['Keeps drinks cold'], ['Stainless steel'], {'Brand':'ExampleCo'})
item = Item(title='Example insulated bottle', price=24.99, category='Home', full=description, weight=get_weight({'Item Weight':'16 ounces'}), id='demo-1')
item.make_prompt(item.full or item.title)
row = {'title':'Desk lamp', 'price':'32.50', 'description':['Adjustable LED desk lamp '*30], 'features':['Dimmable']*30, 'details':{}}
parsed = parse(row, 'Office_Products')
print('Runtime home:', runtime_home(), '| BM25:', settings.bm25_path)
print(item, '| pounds:', item.weight, '| test prompt:', item.test_prompt())
print('Parser produced item:', parsed is not None)

Runtime home: C:\Users\aksha\PycharmProjects\PricerProject | BM25: C:\Users\aksha\PycharmProjects\PricerProject\products_bm25.sqlite3
title='Example insulated bottle' price=24.99 category='Home' full='Example insulated bottle\n[\'Keeps drinks cold\']\n[\'Stainless steel\']\n{"Brand": "ExampleCo"}' weight=1.0 summary=None prompt='What does this cost to the nearest dollar?\n\nExample insulated bottle\n[\'Keeps drinks cold\']\n[\'Stainless steel\']\n{"Brand": "ExampleCo"}\n\nPrice is $25.00' id='demo-1' | pounds: 1.0 | test prompt: What does this cost to the nearest dollar?

Example insulated bottle
['Keeps drinks cold']
['Stainless steel']
{"Brand": "ExampleCo"}

Price is $
Parser produced item: True


## Agent guardrails and retrieval

In [4]:
from pricer.agents.guardrails import extract_single_price, relative_spread
from pricer.agents.retrieval import tokenize, expand_product_query, retrieval_metrics
print('Price extraction:', extract_single_price('Estimated fair price: $28.50'))
print('Estimate spread:', relative_spread([28, 29, 30]))
print('Tokens:', tokenize('Acme X1000 cordless drill'))
print('Query variants:', expand_product_query('the Acme X1000 cordless drill'))
print('Retrieval metrics:', retrieval_metrics(['p1','p2','p3'], {'p2','p3'}, k=3))

Price extraction: 28.5
Estimate spread: 0.06896551724137931
Tokens: ['acme', 'x1000', 'cordless', 'drill']
Query variants: ['the Acme X1000 cordless drill', 'acme x1000 cordless drill', 'acme x1000']
Retrieval metrics: {'precision@3': 0.6666666666666666, 'recall@3': 1.0, 'reciprocal_rank': 0.5, 'ndcg@3': 0.6934264036172708}


## Dataset preparation, baseline models, and evaluation

In [5]:
from pricer.experiments.config import DatasetConfig
from pricer.experiments.data import deduplicate, split_items, lite_splits
from pricer.experiments.baselines import constant_pricer, fit_feature_regression
from pricer.evaluation.evaluator import Tester
products = [Item(title=f'Product {i}', price=p, category='Demo', summary=s, full=s, weight=w) for i,(p,s,w) in enumerate([(20,'small led desk lamp',1),(35,'adjustable reading lamp',2),(15,'insulated water bottle',1),(28,'steel travel bottle',2),(60,'wireless headphones',0),(90,'bluetooth audio headset',0)])]
products = deduplicate(products)
train, validation, holdout = split_items(products, (3,1,2))
mean_model = constant_pricer(train)
feature_model = fit_feature_regression(train)
tester = Tester(feature_model, holdout, size=len(holdout), workers=1)
results = [tester.run_datapoint(i) for i in range(len(holdout))]
mae = sum(result[3] for result in results) / len(results)
print('Dataset:', DatasetConfig(lite=True).repo())
print('Mean baseline prediction:', round(mean_model(holdout[0]), 2))
print('Feature model predictions:', [round(feature_model(x),2) for x in holdout])
print('Holdout MAE:', round(mae,2))
print('Lite split sample:', lite_splits(((1,2),(3,),(4,))))

Dataset: ed-donner/items_lite
Mean baseline prediction: 26.0
Feature model predictions: [9.75, -1.5]
Holdout MAE: 50.88
Lite split sample: ((1, 2), (3,), (4,))


## Persistence, telemetry, and API setup

In [6]:
import tempfile
from pricer.persistence.intelligence_store import IntelligenceStore
from pricer.telemetry.observability import Observability
from pricer.telemetry.monitoring import population_stability_index, drift_level
with tempfile.TemporaryDirectory(prefix='pricer-notebook-') as folder:
    store = IntelligenceStore(Path(folder) / 'demo.sqlite3')
    telemetry = Observability(store)
    with telemetry.trace('notebook.demo', example='local'): answer = extract_single_price('Fair value: $42')
    print('Recorded operation:', answer)
    print('Metrics:', store.metric_summary())
psi = population_stability_index([10,12,13,15,16], [10,11,12,13,14])
print('Drift:', round(psi,4), drift_level(psi))
from pricer.interfaces.api import create_app
from pricer.interfaces.schemas import EstimateRequest
app = create_app()
print('API:', app.title, '| routes:', [r.path for r in app.routes if hasattr(r,'path')][:8])
print('Estimate fields:', list(EstimateRequest.model_fields))

Recorded operation: 42.0
Metrics: [{'name': 'notebook.demo.latency_ms', 'samples': 1, 'average': 5.519599999388447, 'minimum': 5.519599999388447, 'maximum': 5.519599999388447}, {'name': 'notebook.demo.success', 'samples': 1, 'average': 1.0, 'minimum': 1.0, 'maximum': 1.0}]
Drift: 9.7648 drifted
API: Pricer Intelligence API | routes: ['/openapi.json', '/docs', '/docs/oauth2-redirect', '/redoc']
Estimate fields: ['description', 'asking_price', 'url']


## Optional live workflows

These examples are commented out because they need network access, credentials, local model services, or substantial compute. Uncomment only the operation you intend to run.

In [7]:
# Dataset download (network):
# from pricer.data.parser import load_amazon_metadata
# dataset = load_amazon_metadata('Appliances')

# Local or gateway language model (configured service required):
# from pricer.agents.preprocessor import Preprocessor
# summary = Preprocessor().preprocess(item.full or item.title)

# Application deal scanning (interactive/network-dependent):
# from pricer.application.framework import DealAgentFramework
# DealAgentFramework().run()

# Explicit training/index jobs from repository root:
# uv run pricer build-index
# uv run pricer train --epochs 5
# uv run pricer dpo export --dataset artifacts/price_preferences.jsonl
# uv run pricer gateway-eval

The core cells run real project APIs on small local examples. The module inventory also points to the remaining specialized components; use `docs/architecture.md` for production commands and the separate research notebooks for historical QLoRA workflows.